In [3]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import json
import time
import random
import os
from datetime import datetime, timedelta
from tqdm import tqdm
from wakepy import keep


IDS_FILE = os.path.join('/Users/vconklin24/Desktop/data/book_ids/goodreads_ids_temp.csv')
BANNED_FILE = os.path.join('/Users/vconklin24/Desktop/data/datasets/2024-2025_banned_books.csv')
OUTPUT_FILE = os.path.join('/Users/vconklin24/Desktop/data/datasets/reviews_sample.csv')

def parse_date(date_str):
    if pd.isna(date_str): return None
    date_str = str(date_str).replace('Fall', 'September').replace('Spring', 'March')
    try:
        return pd.to_datetime(date_str)
    except:
        return None

def scrape_reviews_for_book(book_url, ban_date, title):
    connector = "&" if "?" in book_url else "?"
    url = f"{book_url}{connector}sort=newest"
    
    headers = {
        'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8',
    }
    
    start_window = ban_date - timedelta(days=180)
    end_window = ban_date + timedelta(days=180)
    reviews_found = []
    
    try:
        response = requests.get(url, headers=headers, timeout=20)
        if response.status_code != 200: return []

        soup = BeautifulSoup(response.content, 'html.parser')
        json_tag = soup.find('script', {'id': '__NEXT_DATA__'})
        
        if json_tag:
            data = json.loads(json_tag.string)
            apollo_state = data.get('props', {}).get('pageProps', {}).get('apolloState', {})
            for key, value in apollo_state.items():
                if key.startswith('Review:'):
                    rev_date_str = value.get('createdAt')
                    if rev_date_str:
                        rev_date = datetime.fromtimestamp(rev_date_str / 1000)
                        if start_window <= rev_date <= end_window:
                            reviews_found.append({
                                'book_title': title,
                                'review_date': rev_date.date(),
                                'rating': value.get('rating'),
                                'review_text': BeautifulSoup(value.get('text', ''), "html.parser").get_text(),
                                'ban_date': ban_date.date()
                            })
        return reviews_found
    except:
        return []

def main():
    print(f"Looking for input files in: {BASE_PATH}")
    if not os.path.exists(IDS_FILE):
        print(f"CRITICAL ERROR: {IDS_FILE} not found!")
        return

    ids_df = pd.read_csv(IDS_FILE)
    banned_df = pd.read_csv(BANNED_FILE)
    
    banned_df['ban_date_dt'] = banned_df['Challenge/Removal'].apply(parse_date)
    top_books = banned_df.groupby(['Title', 'Author']).agg(
        ban_count=('Title', 'count'),
        earliest_ban=('ban_date_dt', 'min')
    ).reset_index().sort_values(by='ban_count', ascending=False).head(20)

    merged = pd.merge(top_books, ids_df, left_on=['Title', 'Author'], right_on=['book_title', 'author'])
    
    final_data = []
    print(f"Starting Scrape for {len(merged)} books...")

    with keep.running():
        for _, row in tqdm(merged.iterrows(), total=merged.shape[0]):
            book_reviews = scrape_reviews_for_book(row['goodreads_url'], row['earliest_ban'], row['Title'])
            if book_reviews:
                final_data.extend(book_reviews)
                print(f"  + Success: {len(book_reviews)} reviews for '{row['Title']}'")
            time.sleep(random.uniform(8, 12))

    if final_data:
        # Saving specifically to the ipynb folder
        pd.DataFrame(final_data).to_csv(OUTPUT_FILE, index=False)
        print(f"\nDONE! File saved to: {OUTPUT_FILE}")
    else:
        print("\nScrape finished but 0 reviews matched your 6-month window.")

if __name__ == "__main__":
    main()

Looking for input files in: /Users/vconklin24/Desktop/data/book_ids
Starting Scrape for 21 books...


  0%|                                                    | 0/21 [00:00<?, ?it/s]

  + Success: 3 reviews for 'Nineteen Minutes'


  5%|██                                          | 1/21 [00:11<03:57, 11.87s/it]

  + Success: 3 reviews for 'Nineteen Minutes'


...

  + Success: 2 reviews for 'The Bluest Eye'


 95%|████████████████████████████████████████▉  | 20/21 [04:27<00:13, 13.57s/it]

  + Success: 1 reviews for 'A Court of Frost and Starlight'


100%|███████████████████████████████████████████| 21/21 [04:39<00:00, 13.29s/it]


DONE! File saved to: /Users/vconklin24/Desktop/data/datasets/reviews_sample.csv


In [5]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import json
import time
import random
import os
from datetime import datetime, timedelta
from tqdm import tqdm
from wakepy import keep

IDS_FILE = os.path.join('/Users/vconklin24/Desktop/data/book_ids/goodreads_ids_temp.csv')
BANNED_FILE = os.path.join('/Users/vconklin24/Desktop/data/datasets/2024-2025_banned_books.csv')
OUTPUT_FILE = os.path.join('/Users/vconklin24/Desktop/data/datasets/reviews_sample.csv')

def parse_date(date_str):
    if pd.isna(date_str): return None
    date_str = str(date_str).replace('Fall', 'September').replace('Spring', 'March')
    try:
        return pd.to_datetime(date_str)
    except:
        return None

def scrape_reviews_for_book(book_url, ban_date, title):
    connector = "&" if "?" in book_url else "?"
    url = f"{book_url}{connector}sort=newest"
    
    headers = {
        'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36',
    }
    
    # WIDENED WINDOW: 1 year before to 1 year after
    start_window = ban_date - timedelta(days=365)
    end_window = ban_date + timedelta(days=365)
    
    reviews_found = []
    fallback_reviews = [] # To store the most recent ones if window fails
    
    try:
        response = requests.get(url, headers=headers, timeout=20)
        if response.status_code != 200: return []
        soup = BeautifulSoup(response.content, 'html.parser')
        json_tag = soup.find('script', {'id': '__NEXT_DATA__'})
        
        if json_tag:
            data = json.loads(json_tag.string)
            apollo_state = data.get('props', {}).get('pageProps', {}).get('apolloState', {})
            
            for key, value in apollo_state.items():
                if key.startswith('Review:'):
                    rev_date_str = value.get('createdAt')
                    if rev_date_str:
                        rev_date = datetime.fromtimestamp(rev_date_str / 1000)
                        
                        review_obj = {
                            'book_title': title,
                            'review_date': rev_date.date(),
                            'rating': value.get('rating'),
                            'review_text': BeautifulSoup(value.get('text', ''), "html.parser").get_text(),
                            'ban_date': ban_date.date(),
                            'match_type': 'In Window'
                        }
                        
                        # Collect for fallback anyway
                        fallback_reviews.append(review_obj)
                        
                        # Check if it fits the window
                        if start_window <= rev_date <= end_window:
                            reviews_found.append(review_obj)

        # FALLBACK: If nothing in the window, take the 5 most recent reviews
        if not reviews_found and fallback_reviews:
            # Sort by date and take top 5
            fallback_reviews.sort(key=lambda x: x['review_date'], reverse=True)
            for r in fallback_reviews[:5]:
                r['match_type'] = 'Fallback (Recent)'
                reviews_found.append(r)
                
        return reviews_found
    except:
        return []

def main():
    ids_df = pd.read_csv(IDS_FILE)
    banned_df = pd.read_csv(BANNED_FILE)
    
    banned_df['ban_date_dt'] = banned_df['Challenge/Removal'].apply(parse_date)
    
    # IMPROVEMENT: Ensure we only have 1 row per book title
    top_books = banned_df.groupby(['Title', 'Author']).agg(
        ban_count=('Title', 'count'),
        earliest_ban=('ban_date_dt', 'min')
    ).reset_index().sort_values(by='ban_count', ascending=False).head(20)

    # Clean IDs file to remove duplicates before merging
    ids_clean = ids_df.drop_duplicates(subset=['book_title', 'author'])

    merged = pd.merge(top_books, ids_clean, left_on=['Title', 'Author'], right_on=['book_title', 'author'])
    
    final_data = []
    print(f"Scraping Top 20 Banned Books (Ensuring 100% coverage)...")

    with keep.running():
        for _, row in tqdm(merged.iterrows(), total=merged.shape[0]):
            book_reviews = scrape_reviews_for_book(row['goodreads_url'], row['earliest_ban'], row['Title'])
            
            if book_reviews:
                final_data.extend(book_reviews)
                print(f"  + Found {len(book_reviews)} reviews for '{row['Title']}'")
            else:
                print(f"  [!] Failed to find ANY reviews for '{row['Title']}'")
            
            time.sleep(random.uniform(7, 10))

    if final_data:
        pd.DataFrame(final_data).to_csv(OUTPUT_FILE, index=False)
        print(f"\nSUCCESS: Data for all reachable books saved to: {OUTPUT_FILE}")

if __name__ == "__main__":
    main()

Scraping Top 20 Banned Books (Ensuring 100% coverage)...


  0%|                                                    | 0/14 [00:00<?, ?it/s]

  + Found 5 reviews for 'Nineteen Minutes'


  7%|███▏                                        | 1/14 [00:09<02:09,  9.94s/it]

  + Found 2 reviews for 'Looking for Alaska'


...

  + Found 6 reviews for 'The Bluest Eye'


 93%|███████████████████████████████████████▉   | 13/14 [02:22<00:10, 10.25s/it]

  + Found 2 reviews for 'A Court of Frost and Starlight'


100%|███████████████████████████████████████████| 14/14 [02:31<00:00, 10.83s/it]



SUCCESS: Data for all reachable books saved to: /Users/vconklin24/Desktop/data/datasets/reviews_sample.csv
